# Chunking Strategy Evaluation - Visualization & Analysis

This notebook provides visualization and analysis of chunking strategy evaluation results.

**Strategies evaluated:**
- **baseline**: Simple character-based chunking
- **sentence**: Sentence-aware chunking  
- **adaptive**: Dynamic strategy selection based on document type

**Usage:**
1. First, run `evaluate_chunking_strategies.py` to generate CSV results
2. Then run this notebook to visualize and analyze the results

All plots will be displayed inline for easy viewing and analysis.

## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure matplotlib for inline plots
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Configuration
RESULTS_DIR = Path("evaluation_results")  # Directory with CSV results from .py script
RESULTS_CSV = RESULTS_DIR / "evaluation_results.csv"
STRATEGY_STATS_CSV = RESULTS_DIR / "strategy_statistics.csv"
NAMESPACE_STATS_CSV = RESULTS_DIR / "namespace_statistics.csv"

# Optional: Enable answer evaluation (requires running evaluation with answer generation)
ENABLE_ANSWER_ANALYSIS = False  # Set to True if you have answer data in results

print("✅ Imports loaded successfully!")
print(f"📁 Working directory: {Path.cwd()}")
print(f"📊 Results directory: {RESULTS_DIR}")


## 2. Load Evaluation Results

In [ ]:
# Load evaluation results from CSV
if not RESULTS_CSV.exists():
    print(f"❌ Results file not found: {RESULTS_CSV}")
    print(f"\n📋 Please run the evaluation script first:")
    print(f"   python evaluate_chunking_strategies.py --output_dir {RESULTS_DIR}")
    raise FileNotFoundError(f"Results file not found: {RESULTS_CSV}")

print(f"📂 Loading results from {RESULTS_CSV}...")
df = pd.read_csv(RESULTS_CSV, encoding="utf-8")

# Load statistics if available
strategy_stats = None
namespace_stats = None

if STRATEGY_STATS_CSV.exists():
    strategy_stats = pd.read_csv(STRATEGY_STATS_CSV, encoding="utf-8")
    print(f"✅ Loaded strategy statistics")

if NAMESPACE_STATS_CSV.exists():
    namespace_stats = pd.read_csv(NAMESPACE_STATS_CSV, encoding="utf-8")
    print(f"✅ Loaded namespace statistics")

print(f"\n✅ Loaded {len(df)} evaluation results")
print(f"📊 Strategies: {', '.join(df['strategy'].unique())}")
print(f"📝 Queries: {df['query'].nunique()}")
print(f"\n📋 Sample data:")
print(df.head())

## 3. Compute Statistics (if not already computed)

In [ ]:
# Compute statistics if not already loaded
if strategy_stats is None:
    print("📊 Computing strategy statistics...")
    strategy_stats = df.groupby("strategy").agg({
        "avg_score": ["mean", "std", "min", "max"],
        "max_score": ["mean", "std"],
        "namespace_correct": "mean",
        "num_results": "mean",
        "unique_docs": "mean",
    }).round(4)
    strategy_stats.columns = ["_".join(col).strip() for col in strategy_stats.columns]
    strategy_stats = strategy_stats.reset_index()
    print("✅ Strategy statistics computed")

if namespace_stats is None:
    print("📊 Computing namespace statistics...")
    namespace_stats = df.groupby(["expected_namespace", "strategy"]).agg({
        "namespace_correct": ["mean", "count"],
    }).round(4)
    namespace_stats.columns = ["accuracy", "count"]
    namespace_stats = namespace_stats.reset_index()
    print("✅ Namespace statistics computed")

# Display statistics
print("\n📈 Strategy Statistics:")
print(strategy_stats.to_string(index=False))
print("\n📈 Namespace Statistics:")
print(namespace_stats.head(10))

## 4. Visualizations - Strategy Comparison

In [ ]:
# Strategy comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Average score comparison
ax = axes[0, 0]
strategy_means = df.groupby("strategy")["avg_score"].mean().sort_values(ascending=False)
bars = ax.bar(strategy_means.index, strategy_means.values, 
              color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_ylabel("Average Retrieval Score", fontsize=11)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_title("Average Retrieval Score by Strategy", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for i, (strategy, val) in enumerate(strategy_means.items()):
    ax.text(i, val + 0.01, f"{val:.3f}", ha="center", va="bottom", fontsize=9)

# Score distribution
ax = axes[0, 1]
for strategy in df["strategy"].unique():
    scores = df[df["strategy"] == strategy]["avg_score"]
    ax.hist(scores, alpha=0.6, label=strategy, bins=15)
ax.set_xlabel("Average Retrieval Score", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("Score Distribution by Strategy", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Namespace accuracy
ax = axes[1, 0]
namespace_acc = df.groupby("strategy")["namespace_correct"].mean()
bars = ax.bar(namespace_acc.index, namespace_acc.values,
              color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_ylabel("Namespace Detection Accuracy", fontsize=11)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_title("Namespace Detection Accuracy by Strategy", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
for i, (strategy, val) in enumerate(namespace_acc.items()):
    ax.text(i, val + 0.02, f"{val:.2%}", ha="center", va="bottom", fontsize=9)

# Unique documents
ax = axes[1, 1]
unique_docs_means = df.groupby("strategy")["unique_docs"].mean()
bars = ax.bar(unique_docs_means.index, unique_docs_means.values,
              color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_ylabel("Average Unique Documents", fontsize=11)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_title("Document Diversity by Strategy", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for i, (strategy, val) in enumerate(unique_docs_means.items()):
    ax.text(i, val + 0.1, f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "strategy_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved plot to {RESULTS_DIR / 'strategy_comparison.png'}")

## 5. Visualization - Namespace Accuracy Heatmap

In [ ]:
# Namespace accuracy heatmap
pivot = namespace_stats.pivot(index="expected_namespace", columns="strategy", values="accuracy")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt=".2%", cmap="YlOrRd",
            cbar_kws={"label": "Accuracy"}, ax=ax)
ax.set_title("Namespace Detection Accuracy\n(Expected vs. Detected)", 
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_ylabel("Expected Namespace", fontsize=11)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "namespace_accuracy_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved plot to {RESULTS_DIR / 'namespace_accuracy_heatmap.png'}")

## 6. Visualization - Category Analysis

In [ ]:
# Category analysis
category_strategy = df.groupby(["category", "strategy"])["avg_score"].mean().reset_index()
pivot = category_strategy.pivot(index="category", columns="strategy", values="avg_score")

fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e", "#2ca02c"], width=0.8)
ax.set_ylabel("Average Retrieval Score", fontsize=11)
ax.set_xlabel("Query Category", fontsize=11)
ax.set_title("Retrieval Performance by Query Category and Strategy", 
             fontsize=12, fontweight="bold")
ax.legend(title="Strategy", title_fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "category_analysis.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved plot to {RESULTS_DIR / 'category_analysis.png'}")

## 7. Summary and Conclusions

In [ ]:
# Find best strategy
best_strategy = strategy_stats.loc[strategy_stats["avg_score_mean"].idxmax(), "strategy"]
best_avg_score = strategy_stats["avg_score_mean"].max()
overall_namespace_acc = df["namespace_correct"].mean()

print("=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)
print(f"\n✅ Best Performing Strategy (by retrieval score): {best_strategy}")
print(f"   Average Retrieval Score: {best_avg_score:.4f}")
print(f"\n✅ Overall Namespace Detection Accuracy: {overall_namespace_acc:.2%}")

# Category-specific recommendations
print(f"\n📊 Performance by Category:")
category_best = df.groupby(["category", "strategy"])["avg_score"].mean().reset_index()
category_best = category_best.loc[category_best.groupby("category")["avg_score"].idxmax()]
for _, row in category_best.iterrows():
    print(f"  {row['category']}: {row['strategy']} (score: {row['avg_score']:.4f})")

print(f"\n✅ All visualizations saved to: {RESULTS_DIR}")
print("=" * 70)

## 8. Additional Analysis (Optional)

In [ ]:
# Additional analysis: Query-level performance
print("📊 Top performing queries by strategy:")
for strategy in df["strategy"].unique():
    strategy_df = df[df["strategy"] == strategy]
    top_queries = strategy_df.nlargest(3, "avg_score")[["query", "avg_score"]]
    print(f"\n{strategy.upper()}:")
    for _, row in top_queries.iterrows():
        print(f"  {row['query'][:50]}... (score: {row['avg_score']:.4f})")

# Additional analysis: Worst performing queries
print("\n\n📊 Queries with lowest scores:")
worst_queries = df.nsmallest(5, "avg_score")[["query", "strategy", "avg_score"]]
for _, row in worst_queries.iterrows():
    print(f"  {row['strategy']}: {row['query'][:50]}... (score: {row['avg_score']:.4f})")

## 9. Answer Quality Analysis (if available)

In [ ]:
# Analyze answer quality if answers were generated
if ENABLE_ANSWER_ANALYSIS and "llm_score" in df.columns and "answer" in df.columns:
    print("📊 Answer Quality Analysis:")
    print("=" * 70)
    
    # LLM score by strategy
    llm_by_strategy = df.groupby("strategy")["llm_score"].agg(["mean", "std", "count"]).round(4)
    print("\nLLM Answer Quality Score by Strategy:")
    print(llm_by_strategy)
    
    # Precision/Recall if available
    if "precision_at_k" in df.columns and "recall_at_k" in df.columns:
        pr_by_strategy = df.groupby("strategy")[["precision_at_k", "recall_at_k"]].mean().round(4)
        print("\nPrecision and Recall by Strategy:")
        print(pr_by_strategy)
        
        # Plot precision/recall
        fig, ax = plt.subplots(figsize=(10, 6))
        pr_by_strategy.plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e"])
        ax.set_ylabel("Score", fontsize=11)
        ax.set_xlabel("Chunking Strategy", fontsize=11)
        ax.set_title("Precision and Recall by Strategy", fontsize=12, fontweight="bold")
        ax.legend(title="Metric")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "precision_recall_by_strategy.png", dpi=300, bbox_inches="tight")
        plt.show()
        print(f"✅ Saved plot to {RESULTS_DIR / 'precision_recall_by_strategy.png'}")
    
    # Plot LLM scores
    fig, ax = plt.subplots(figsize=(10, 6))
    llm_means = df.groupby("strategy")["llm_score"].mean().sort_values(ascending=False)
    bars = ax.bar(llm_means.index, llm_means.values, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
    ax.set_ylabel("Average LLM Quality Score", fontsize=11)
    ax.set_xlabel("Chunking Strategy", fontsize=11)
    ax.set_title("Answer Quality by Strategy (LLM Grading)", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)
    for i, (strategy, val) in enumerate(llm_means.items()):
        ax.text(i, val + 0.02, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "llm_score_by_strategy.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved plot to {RESULTS_DIR / 'llm_score_by_strategy.png'}")
    
else:
    print("ℹ️ Answer evaluation data not available in results.")
    print("   To enable answer evaluation, modify the evaluation script to generate answers.")